# Chapter 15 — Supplement 2: `extract_facts`

*The model proposes, deterministic rules dispose.*

Companion to `15_capstone.ipynb`. `extract_facts` is the one workflow tool whose job — turn free text into a few structured fields — is a natural fit for a language model, and also the one place a brittle implementation quietly costs us downstream. This notebook **calls the shipped hybrid implementation** and shows why it pairs an LLM with a deterministic guard.

## Where it fits

Step 2 of the workflow. It returns `{product, issue, urgency, sentiment, summary}`. Its `issue` field is what `flag_regulatory` keys on: the UDAAP rule fires only when `issue` is `overdraft_fee`. So extraction errors become **silent escalation errors** two steps later.

| | |
| --- | --- |
| **Backed by** | Qwen2.5-3B-Instruct (proposes) + a deterministic normalizer & guards (dispose) |
| **Artifact** | none trained — wraps the off-the-shelf instruct model |
| **Module** | `glassloop/models/qwen_extractor.py` + `glassloop/capstone/banking_tools.py` |
| **Offline toggle** | `AGENTLAB_USE_LLM_EXTRACT=0` forces the pure-rule path |
| **Risk level** | `LOW` |

The chapter (§"Fact extraction with a small open-weight LLM") frames the whole point: *putting a language model inside a governed workflow is safe exactly when a deterministic gate decides what the model's output is allowed to cause.*

In [ ]:
import os, json, warnings, contextlib, io
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('KNOWLYTIX_EULA_ACCEPTED', '1')  # silence the EULA reminder line
warnings.filterwarnings('ignore')  # quiet the harmless GPU-capability / HF notices

# knowlytix prints a one-line license banner (it names the licensee) to stderr
# on first import. The package has no flag to disable it, so import it once here
# with stderr/stdout captured; every later import is a cache hit and stays quiet,
# so the banner never lands in a baked cell output.
with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    try:
        import knowlytix.core  # noqa: F401
    except Exception:
        pass

root = Path('.')
if not (root / 'data').exists():
    root = Path('..')
assert (root / 'data').exists(), 'run this notebook from the repo root or notebooks/'
print('repo root:', root.resolve())

## 1. The LLM half — Qwen proposes

`get_default_extractor()` loads Qwen2.5-3B-Instruct once and asks it, per message, for a JSON object with exactly four keys. Greedy decoding makes the output deterministic. On any failure (load error, unparseable output) it returns `None` — the tool then degrades to the rule extractor, so the pipeline runs even with no GPU.

In [ ]:
from glassloop.models.qwen_extractor import get_default_extractor

extract = get_default_extractor()
for msg in [
    'I was charged twice for the same purchase, please reverse one.',   # case-020
    "I'm going to sue you unless you give me my money back today.",      # case-009
]:
    print(msg)
    print('   raw LLM ->', extract.extract(msg))
    print()

**Reading the output.** The model returns clean four-field JSON. But the raw output is *not constrained to our taxonomy* and cannot be trusted to drive a regulated decision directly. The classic failure: case-009 ("give me my money back") names no product and no fee, yet a helpful model can pattern-match "money back" into a `checking_account / overdraft_fee` reading — which would trip UDAAP and force a **false escalation**. Dropped in naively, that single case takes escalation accuracy from 100% to 95%.

## 2. The deterministic half — rules dispose

`_normalize_llm` coerces the model's free-text `product` onto the five enumerable values (falling back to the keyword result when the model goes off-taxonomy), derives `issue` from that validated product, and then enforces two **evidence guards**: *a regulatory issue may only fire on evidence present in the source text.*

In [ ]:
from glassloop.capstone.banking_tools import (
    _rule_extract, _normalize_llm, _FEE_SIGNAL, _REGX_SIGNAL, _PRODUCTS,
)

print('enumerable products :', _PRODUCTS)
print('UDAAP fee signal    :', _FEE_SIGNAL.pattern)
print('Reg-X loan signal   :', _REGX_SIGNAL.pattern)

The **UDAAP guard** demotes `overdraft_fee` back to `account_issue` unless the message actually contains a fee / charge / dollar amount / reversal token. The **Reg-X guard** discards a `mortgage`/`loan` product unless the message names one. Below we feed the normalizer a deliberately over-reaching LLM proposal for case-009 and watch the guard fire — this is deterministic, so it behaves identically every run:

In [ ]:
msg = "I'm going to sue you unless you give me my money back today."
rule = _rule_extract(msg)
# Simulate an LLM that over-reached to overdraft_fee with no fee evidence in the text:
over_reaching = {'product': 'checking_account', 'issue': 'overdraft_fee',
                 'urgency': 'high', 'sentiment': 'negative'}
guarded = _normalize_llm(over_reaching, msg, rule)

print('message contains a fee signal? ', bool(_FEE_SIGNAL.search(msg)))
print('LLM proposed issue           : ', over_reaching['issue'])
print('guarded issue                : ', guarded['issue'])
assert guarded['issue'] != 'overdraft_fee', 'guard should have demoted it'
print('\nUDAAP guard fired -> no false escalation on case-009')

Because the message carries no fee token, the guard demotes `overdraft_fee` to `account_issue`; UDAAP stays silent and escalation accuracy returns to 100%. The model still contributes its recall on the cases that *do* carry a fee signal — it simply cannot manufacture a regulatory event out of thin air.

## 3. The hybrid tool end to end

`extract_facts` (the registered `Tool`) wires both halves: always compute the rule baseline, ask the LLM, and run `_normalize_llm` over the result. Here is the governed tool producing its final, taxonomy-safe output.

In [ ]:
from glassloop.capstone.banking_tools import extract_facts

for msg in [
    'I was charged a $35 overdraft fee on my checking account, reverse it!',
    "I'm going to sue you unless you give me my money back today.",   # case-009
    'There is an escrow error on my home mortgage statement.',
]:
    out = extract_facts.fn(**extract_facts.input_schema(message=msg).model_dump())
    print(msg)
    print('   product =', out['product'], '| issue =', out['issue'],
          '| urgency =', out['urgency'], '| sentiment =', out['sentiment'])
    print()

The first message keeps `overdraft_fee` (the fee signal is present, so UDAAP *should* fire); case-009 lands on `account_issue` (guarded); the mortgage message keeps a mortgage product because the Reg-X signal is present. The final dict is always inside the enumerable taxonomy, whatever the model said.

## 4. The offline / deterministic toggle

The test suite runs with `AGENTLAB_USE_LLM_EXTRACT=0` to stay fast, offline, and fully deterministic. With the flag set, `extract_facts` skips the model entirely and returns the pure keyword extractor's result — the degrade-safe fallback path.

In [ ]:
import os
msg = 'I was charged a $35 overdraft fee on my checking account, reverse it!'

os.environ['AGENTLAB_USE_LLM_EXTRACT'] = '0'   # force the rule-only path
rule_only = extract_facts.fn(**extract_facts.input_schema(message=msg).model_dump())
os.environ['AGENTLAB_USE_LLM_EXTRACT'] = '1'   # restore the hybrid path
hybrid = extract_facts.fn(**extract_facts.input_schema(message=msg).model_dump())

print('rule-only :', rule_only['product'], '/', rule_only['issue'])
print('hybrid    :', hybrid['product'], '/', hybrid['issue'])

On a message the keyword ladder *can* handle, both paths agree. The hybrid's advantage shows on phrasing the ladder cannot enumerate (a fee complaint that never says the literal word "overdraft") — there the LLM recovers recall the rules miss, while the guards keep it honest.

## Summary

- `extract_facts` = **LLM proposes, deterministic rules dispose**. The LLM adds recall over open-ended phrasing; `_normalize_llm` keeps every output inside the taxonomy.
- Two evidence guards (`_FEE_SIGNAL`, `_REGX_SIGNAL`) forbid a regulatory issue the source text does not support — the case-009 UDAAP false-escalation fix.
- `AGENTLAB_USE_LLM_EXTRACT=0` gives a GPU-free, deterministic fallback.

Next: **Supplement 3 — `search_policy`**, Graph RAG that retrieves on a graph and generates on the model.